# Análisis del Mercado de Criptomonedas

Exploración de datos en tiempo real desde CoinGecko API para las principales criptomonedas: Bitcoin, Ethereum, Solana, Cardano y Ripple.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import requests
import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
coins = ["bitcoin", "ethereum", "solana", "cardano", "ripple"]
vs_currency = "usd"

market_url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": vs_currency,
    "ids": ",".join(coins),
    "order": "market_cap_desc",
    "per_page": len(coins),
    "sparkline": "false",
    "price_change_percentage": "24h,7d,30d",
}
resp = requests.get(market_url, params=params, timeout=30)
market = pd.DataFrame(resp.json())

history_frames = []
for coin in coins[:3]:
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart"
    params = {"vs_currency": vs_currency, "days": 90, "interval": "daily"}
    resp = requests.get(url, params=params, timeout=30)
    data = resp.json()
    prices = pd.DataFrame(data["prices"], columns=["timestamp", "price"])
    prices["date"] = pd.to_datetime(prices["timestamp"], unit="ms").dt.date
    prices["coin"] = coin
    history_frames.append(prices)
    time.sleep(1.5)

history = pd.concat(history_frames, ignore_index=True)

print(f"Monedas cargadas: {len(market)}")
print(f"Registros históricos: {len(history)}")
market[["name", "current_price", "market_cap", "total_volume"]].to_string()

## Resumen del mercado

In [ ]:
resumen = market[["name", "current_price", "market_cap", "total_volume",
                   "price_change_percentage_24h_in_currency",
                   "price_change_percentage_7d_in_currency",
                   "price_change_percentage_30d_in_currency"]].copy()
resumen.columns = ["Moneda", "Precio USD", "Market Cap", "Volumen 24h", "Cambio 24h %", "Cambio 7d %", "Cambio 30d %"]
resumen

## 1. Capitalización de mercado por moneda

In [ ]:
paleta = ["#0D1B2A", "#1B2838", "#415A77", "#778DA9", "#E0E1DD"]

market_sorted = market.sort_values("market_cap", ascending=True).copy()
market_sorted["cap_billions"] = market_sorted["market_cap"] / 1e9

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(market_sorted["name"], market_sorted["cap_billions"],
               color=paleta[:len(market_sorted)], edgecolor="none", height=0.6)

for bar, valor in zip(bars, market_sorted["cap_billions"]):
    ax.text(bar.get_width() + bar.get_width() * 0.02, bar.get_y() + bar.get_height() / 2,
            f"${valor:,.1f}B", va="center", fontsize=10, color="#333")

ax.set_title("Capitalización de mercado actual", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Market Cap (USD Billones)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${x:,.0f}B"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Correlación de cambios de precio

In [ ]:
cambios = market[["name",
                   "price_change_percentage_24h_in_currency",
                   "price_change_percentage_7d_in_currency",
                   "price_change_percentage_30d_in_currency"]].copy()
cambios.columns = ["moneda", "24h", "7d", "30d"]
cambios = cambios.set_index("moneda")

corr = cambios.T.corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, vmin=-1, vmax=1, square=True, linewidths=0.8,
    cbar_kws={"shrink": 0.75, "label": "Correlación"},
    ax=ax
)
ax.set_title("Correlación de cambios de precio entre criptomonedas", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

## 3. Tendencia de precios normalizada (90 días)

In [ ]:
colores_linea = {"bitcoin": "#0D1B2A", "ethereum": "#415A77", "solana": "#778DA9"}
nombres_display = {"bitcoin": "Bitcoin", "ethereum": "Ethereum", "solana": "Solana"}

fig, ax = plt.subplots(figsize=(12, 5))

for coin in history["coin"].unique():
    df_coin = history[history["coin"] == coin].sort_values("date").copy()
    precio_base = df_coin["price"].iloc[0]
    df_coin["normalizado"] = (df_coin["price"] / precio_base) * 100
    ax.plot(df_coin["date"], df_coin["normalizado"], color=colores_linea[coin],
            linewidth=2, label=nombres_display[coin])

ax.axhline(y=100, color="#999", linestyle=":", linewidth=0.8, alpha=0.5)
ax.set_title("Rendimiento normalizado (base 100) - Últimos 90 días", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Índice (inicio = 100)")
ax.set_xlabel("")
ax.legend(fontsize=10, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Precios históricos interactivos

In [ ]:
colores_plotly = {"bitcoin": "#0D1B2A", "ethereum": "#415A77", "solana": "#778DA9"}

fig = px.line(
    history, x="date", y="price", color="coin",
    color_discrete_map=colores_plotly,
    labels={"price": "Precio (USD)", "date": "Fecha", "coin": "Moneda"},
    title="Precio histórico - Últimos 90 días"
)
fig.update_traces(hovertemplate="%{x}<br>$%{y:,.2f}<extra>%{fullData.name}</extra>")
fig.update_layout(
    template="plotly_white",
    font=dict(family="Inter, sans-serif"),
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    yaxis=dict(tickprefix="$", tickformat=",.0f")
)
fig.show()

## 5. Dominancia de mercado (treemap)

In [ ]:
market_tree = market[["name", "market_cap", "current_price"]].copy()
total_cap = market_tree["market_cap"].sum()
market_tree["dominancia_pct"] = (market_tree["market_cap"] / total_cap * 100).round(2)
market_tree["etiqueta"] = market_tree.apply(
    lambda r: f"{r['name']}<br>${r['current_price']:,.0f}<br>{r['dominancia_pct']:.1f}%", axis=1
)

fig = px.treemap(
    market_tree, path=["name"], values="market_cap",
    color="dominancia_pct",
    color_continuous_scale=["#E0E1DD", "#778DA9", "#415A77", "#1B2838", "#0D1B2A"],
    title="Dominancia de mercado entre criptomonedas seleccionadas",
    custom_data=["etiqueta"]
)
fig.update_traces(
    texttemplate="%{customdata[0]}",
    hovertemplate="%{label}<br>Market Cap: $%{value:,.0f}<extra></extra>"
)
fig.update_layout(
    template="plotly_white",
    font=dict(family="Inter, sans-serif"),
    height=480,
    coloraxis_colorbar=dict(title="Dominancia %")
)
fig.show()

## Hallazgos principales

- Bitcoin domina el mercado con más del 60% de la capitalización entre las monedas analizadas
- Las correlaciones de precio muestran movimientos sincronizados entre BTC y ETH
- Solana presenta mayor volatilidad relativa en el período de 90 días
- El treemap evidencia la concentración extrema del mercado crypto en las dos monedas principales